# Panel Regression Analysis
**Proyecto:** Futuro de las Empresas de Software ante la IA Generativa  
**Fase 2:** Modelos econometricos - OLS con efectos fijos, DiD, tests de robustez  
**Autor:** Alejandro Jimenez  
**Ultima actualizacion:** 2026-05-17

## Especificacion del modelo principal
$$\ln(\text{Return}_{it}) = \alpha_i + \gamma_t + \beta_1 \cdot \text{AI\_intensity}_{it} + \beta_2 \cdot \text{TechDebt}_{it} + \beta_3 \cdot \ln(\text{Revenue}_{it}) + \varepsilon_{it}$$

Donde $\alpha_i$ = efectos fijos por empresa, $\gamma_t$ = efectos fijos de tiempo.

## Hipotesis
- **H1:** Empresas con mayor AI intensity tienen mayores retornos (beta_1 > 0)
- **H2:** La deuda tecnica modera negativamente el efecto AI (beta_2 < 0)
- **H3:** El evento ChatGPT (nov 2022) tiene efecto diferencial para tratados (DiD)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, BetweenOLS
from linearmodels.panel import compare
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print('Paquetes cargados correctamente')

## 1. Carga y preparacion del panel

In [ ]:
from pathlib import Path

BASE = Path('..') / 'data'
panel = pd.read_csv(BASE / 'processed' / 'panel_final.csv')

print(f'Panel cargado: {panel.shape}')
print(f'Empresas: {panel["ticker"].nunique()}')
print(f'Anos: {sorted(panel["year"].unique())}')
panel.head()

In [ ]:
# Preparar variables para regresion
df = panel.copy()

# Variable dependiente: IHS (inverse hyperbolic sine) maneja ceros y negativos
df['return_ihs'] = np.arcsinh(df['return_annual_pct'])

# log_revenues ya viene de build_panel.py; recalcular por si acaso
df['log_revenues'] = np.log(df['revenues'].clip(lower=1))

# Winsorizar tech_debt_proxy en p95 para eliminar outliers estructurales
# SOUN 2022-23: COGS/Rev >1350% (startup pre-rentabilidad); PRGS 2021: write-down XBRL artifical
p95_tech = df['tech_debt_proxy'].quantile(0.95)
df['tech_debt_proxy_w'] = df['tech_debt_proxy'].clip(upper=p95_tech)
print(f'tech_debt_proxy winsorizado en p95 = {p95_tech:.1f}%')

# Dataset 1: todas las obs con retorno + ai_intensity (N=437)
key_vars = ['return_annual_pct', 'ai_intensity', 'year', 'ticker']
df_clean = df.dropna(subset=key_vars).copy()
print(f'df_clean (modelo base):     {len(df_clean)} obs, {df_clean["ticker"].nunique()} tickers')

# Dataset 2: subconjunto con fundamentales EDGAR (revenues + cogs -> tech_debt_proxy)
df_full = df.dropna(subset=['return_annual_pct', 'ai_intensity', 'tech_debt_proxy_w', 'log_revenues']).copy()
print(f'df_full  (modelo completo): {len(df_full)} obs, {df_full["ticker"].nunique()} tickers')
print(f'  Cobertura EDGAR: tech_debt_proxy para 46 empresas (v3: CIKs corregidos + IFRS support)')

In [ ]:
# Estadisticas descriptivas
vars_desc = ['return_annual_pct', 'return_ihs', 'ai_intensity', 'tech_debt_proxy', 
             'rd_intensity', 'op_margin', 'net_margin', 'ai_usage_pct']
available = [v for v in vars_desc if v in df_clean.columns]

desc = df_clean[available].describe().T
desc['cv'] = desc['std'] / desc['mean'].abs()  # coeficiente de variacion
print('\nEstadisticas descriptivas:')
desc.round(3)

In [ ]:
# Distribucion por grupo
print('Retornos por grupo de empresa:')
df_clean.groupby('group')['return_annual_pct'].agg(['mean', 'median', 'std', 'count']).round(2)

## 2. Modelo de panel con efectos fijos (PanelOLS)

Usando `linearmodels.PanelOLS` que implementa efectos fijos de dos vias (empresa + ano) con SE robustos cluster-level.

In [ ]:
# Configurar indice de panel (MultiIndex: entidad, tiempo)
def make_panel(df, entity='ticker', time='year'):
    d = df.copy()
    d = d.set_index([entity, time])
    # linearmodels requiere que el indice sea categorico para efectos fijos
    return d

# Modelo 1: Solo AI intensity (sin controles)
panel_m1 = make_panel(df_clean)

mod1 = PanelOLS(
    panel_m1['return_ihs'],
    sm.add_constant(panel_m1[['ai_intensity']]),
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res1 = mod1.fit(cov_type='clustered', cluster_entity=True)
print('=== Modelo 1: Return ~ AI_intensity (EF empresa + ano) ===')
print(res1.summary.tables[1])

In [ ]:
# Modelo 2: AI intensity + Tech Debt (winsorizado) + log(Revenue)
panel_m2 = make_panel(df_full)

exog2 = sm.add_constant(panel_m2[['ai_intensity', 'tech_debt_proxy_w', 'log_revenues']])

mod2 = PanelOLS(
    panel_m2['return_ihs'],
    exog2,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res2 = mod2.fit(cov_type='clustered', cluster_entity=True)
print(f'=== Modelo 2: Return ~ AI_intensity + TechDebt_w + ln(Revenue) (N={int(mod2.fit().nobs)}, 32 tickers) ===')
print(res2.summary.tables[1])

In [ ]:
# Modelo 3: + rd_intensity como control de innovacion
df_m3 = df.dropna(subset=['return_annual_pct', 'ai_intensity', 'tech_debt_proxy_w', 'log_revenues', 'rd_intensity']).copy()
panel_m3 = make_panel(df_m3)

exog3 = sm.add_constant(panel_m3[['ai_intensity', 'tech_debt_proxy_w', 'log_revenues', 'rd_intensity']])

mod3 = PanelOLS(
    panel_m3['return_ihs'],
    exog3,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res3 = mod3.fit(cov_type='clustered', cluster_entity=True)
print(f'=== Modelo 3: + rd_intensity (N={int(mod3.fit().nobs)}) ===')
print(res3.summary.tables[1])

In [ ]:
# Tabla comparativa de modelos
print('=== Comparacion de modelos ===')
try:
    comp = compare({'Modelo 1': res1, 'Modelo 2': res2, 'Modelo 3': res3})
    print(comp)
except Exception as e:
    # Si los modelos tienen distintos N, mostrar tabla manual
    rows = []
    for name, res in [('M1', res1), ('M2', res2), ('M3', res3)]:
        rows.append({
            'Model': name,
            'N': int(res.nobs),
            'R2_within': round(res.rsquared, 4),
            'F_stat': round(res.f_statistic.stat, 3) if hasattr(res.f_statistic, 'stat') else None
        })
    print(pd.DataFrame(rows).to_string(index=False))
    print(f'Note: {e}')

## 3. Test de Hausman: efectos fijos vs. aleatorios

H0: Los efectos individuales no estan correlacionados con los regresores (efectos aleatorios son eficientes)  
H1: Los efectos individuales estan correlacionados (solo efectos fijos son consistentes)

In [ ]:
# Hausman test usando df_full (N=180, 32 tickers — v3 con CIKs corregidos)
re_mod = RandomEffects(
    panel_m2['return_ihs'],
    sm.add_constant(panel_m2[['ai_intensity', 'tech_debt_proxy_w', 'log_revenues']])
)
res_re = re_mod.fit()

b_fe = res2.params[['ai_intensity', 'tech_debt_proxy_w', 'log_revenues']]
b_re = res_re.params[['ai_intensity', 'tech_debt_proxy_w', 'log_revenues']]

diff = b_fe - b_re
var_fe = np.diag(res2.cov.loc[b_fe.index, b_fe.index].values)
var_re = np.diag(res_re.cov.loc[b_re.index, b_re.index].values)
var_diff = var_fe - var_re

hausman_stat = float(diff.values @ np.linalg.pinv(np.diag(var_diff)) @ diff.values)
hausman_df = len(diff)
hausman_p = 1 - stats.chi2.cdf(hausman_stat, df=hausman_df)

print(f'Test de Hausman (N=180, subconjunto con fundamentales EDGAR)')
print(f'  Chi2({hausman_df}) = {hausman_stat:.4f}')
print(f'  p-value          = {hausman_p:.4f}')
print()
if hausman_p < 0.05:
    print('  -> Rechazar H0: usar Efectos FIJOS (FE consistentes)')
else:
    print('  -> No rechazar H0: Efectos Aleatorios preferibles')
    print('  Interpretacion: la variacion ENTRE empresas explica mas que la variacion DENTRO.')
    print('  Las empresas con tech_debt/revenues estructuralmente distintos difieren en retornos.')
    print('  -> Se reportan ambos (FE y RE); se discute en seccion de robustez.')

In [ ]:
from linearmodels.panel import BetweenOLS

# Between-OLS: promedio por empresa a lo largo del tiempo
be_mod = BetweenOLS(
    panel_m2['return_ihs'],
    sm.add_constant(panel_m2[['ai_intensity', 'tech_debt_proxy_w', 'log_revenues']])
)
res_be = be_mod.fit(cov_type='robust')

print('=== Between-OLS: variacion ENTRE empresas ===')
print(res_be.summary.tables[1])

# Efectos Aleatorios (combina within + between)
print()
print('=== Random Effects (GLS) ===')
print(res_re.summary.tables[1])

# Tabla comparativa FE vs RE vs Between
print()
print(f'{"":30} {"FE (within)":>15} {"RE (GLS)":>15} {"Between":>15}')
print('-' * 75)
for v in ['ai_intensity', 'tech_debt_proxy_w', 'log_revenues']:
    fe_c  = f'{res2.params[v]:.4f}' + ('*' if res2.pvalues[v] < 0.10 else '')
    re_c  = f'{res_re.params[v]:.4f}' + ('*' if res_re.pvalues[v] < 0.10 else '')
    be_c  = f'{res_be.params[v]:.4f}' + ('*' if res_be.pvalues[v] < 0.10 else '')
    print(f'{v:30} {fe_c:>15} {re_c:>15} {be_c:>15}')
print(f'{"N":30} {int(res2.nobs):>15} {int(res_re.nobs):>15} {int(res_be.nobs):>15}')
print(f'{"R2":30} {res2.rsquared:>14.4f} {res_re.rsquared:>14.4f} {res_be.rsquared:>14.4f}')
print()
print('Nota: * p<0.10. FE elimina efectos invariantes en el tiempo (entre empresas).')
print('Between-OLS captura diferencias estructurales entre empresas (mas relevante para H2).')

## 3b. Between-OLS y Efectos Aleatorios (alternativa al FE cuando Hausman no rechaza)

Between-OLS captura la variacion ENTRE empresas (promedios por empresa), que es la dimensión relevante para preguntas cross-seccionales como "empresas con mas deuda tecnica tienen peores retornos?"

## 4. Difference-in-Differences (DiD) — Evento ChatGPT (Nov 2022)

Especificacion:
$$\text{Return}_{it} = \alpha_i + \gamma_t + \delta \cdot (\text{Treated}_i \times \text{Post}_{t}) + X_{it}\Gamma + \varepsilon_{it}$$

- **Treated:** empresa que lanzo producto AI generativa en 2022-2023 (ventana 1 ano del evento ChatGPT)
- **Post:** 1 si ano > 2022

In [ ]:
# Verificar distribucion treated
print('Empresas tratadas (treated_chatgpt=1):')
treated_firms = df_clean[df_clean['treated_chatgpt']==1]['ticker'].unique()
print(treated_firms)
print(f'N tratados: {len(treated_firms)}')
print(f'N control:  {df_clean[df_clean["treated_chatgpt"]==0]["ticker"].nunique()}')

In [ ]:
# Crear interaccion DiD
df_did = df_clean.copy()
df_did['did_chatgpt'] = df_did['treated_chatgpt'] * df_did['post_chatgpt']

panel_did = make_panel(df_did)

# DiD basico
exog_did = panel_did[['did_chatgpt', 'treated_chatgpt']]
exog_did = sm.add_constant(exog_did)

mod_did = PanelOLS(
    panel_did['return_ihs'],
    exog_did,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_did = mod_did.fit(cov_type='clustered', cluster_entity=True)
print('=== DiD basico: Return ~ did_chatgpt (EF empresa + ano) ===')
print(res_did.summary.tables[1])

In [ ]:
# DiD con controles
df_did2 = df_did.dropna(subset=['ai_intensity']).copy()
panel_did2 = make_panel(df_did2)

exog_did2 = panel_did2[['did_chatgpt', 'treated_chatgpt', 'ai_intensity']]
exog_did2 = sm.add_constant(exog_did2)

mod_did2 = PanelOLS(
    panel_did2['return_ihs'],
    exog_did2,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_did2 = mod_did2.fit(cov_type='clustered', cluster_entity=True)
print('=== DiD con controles: + AI intensity ===')
print(res_did2.summary.tables[1])

did_coef = res_did2.params.get('did_chatgpt', None)
did_pval = res_did2.pvalues.get('did_chatgpt', None)
if did_coef is not None:
    print(f'\nEfecto DiD (delta): {did_coef:.4f} (p={did_pval:.4f})')
    print('Interpretacion: diferencia en retornos IHS entre tratados y controles post-ChatGPT')

## 5. Test de Parallel Trends (pre-requisito del DiD)

Verificamos que antes del evento (2022), tratados y controles tenian tendencias paralelas en retornos.

In [ ]:
# Datos pre-evento (antes de 2022)
pre_event = df_clean[df_clean['year'] < 2022].copy()

# Regresion: Return ~ year * treated (pre-periodo)
# Si el coeficiente de year*treated no es significativo, hay parallel trends
pre_event['year_centered'] = pre_event['year'] - 2021  # centrar en 2021
pre_event['trend_interaction'] = pre_event['year_centered'] * pre_event['treated_chatgpt']

panel_pre = make_panel(pre_event)
exog_pre = panel_pre[['trend_interaction', 'treated_chatgpt']]
exog_pre = sm.add_constant(exog_pre)

mod_pre = PanelOLS(
    panel_pre['return_ihs'],
    exog_pre,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_pre = mod_pre.fit(cov_type='clustered', cluster_entity=True)
print('=== Test de Parallel Trends (pre-2022) ===')
print(res_pre.summary.tables[1])

pt_coef = res_pre.params.get('trend_interaction', None)
pt_pval = res_pre.pvalues.get('trend_interaction', None)
if pt_coef is not None:
    print(f'\nCoeficiente trend_interaction: {pt_coef:.4f} (p={pt_pval:.4f})')
    if pt_pval > 0.05:
        print('-> No rechazar H0: supuesto de tendencias paralelas se CUMPLE')
    else:
        print('-> ATENCION: tendencias paralelas posiblemente violadas')

## 6. Event Study — Retornos por ano relativo al evento

In [ ]:
# Retorno medio por ano, separado por grupo treated/control
event_year = 2022
df_event = df_clean.copy()
df_event['event_time'] = df_event['year'] - event_year
df_event['group_label'] = df_event['treated_chatgpt'].map({1: 'Tratados', 0: 'Control'})

event_study = df_event.groupby(['event_time', 'group_label'])['return_annual_pct'].agg(['mean', 'sem', 'count'])
event_study.columns = ['mean_return', 'se_return', 'n']
event_study['ci_lower'] = event_study['mean_return'] - 1.96 * event_study['se_return']
event_study['ci_upper'] = event_study['mean_return'] + 1.96 * event_study['se_return']

print('Retornos medios por ano relativo al evento ChatGPT (2022):')
print(event_study.round(2).to_string())

In [ ]:
# Guardar datos para grafico de event study
event_study.reset_index().to_csv('../data/processed/event_study_chatgpt.csv', index=False)
print('Datos de event study guardados')

## 7. Leave-One-Out: analisis de outliers

NVDA y tickers de infra pueden dominar los resultados por el ciclo de GPU/AI. Verificamos la robustez excluyendolos.

In [ ]:
outlier_tickers = ['NVDA', 'AMD', 'SMCI', 'PLTR']  # tickers de alto impacto potencial

results_loo = {}

# Modelo base
results_loo['full'] = res1

# Leave-one-out por grupo
df_no_infra = df_clean[df_clean['group'] != 'infra'].copy()
if len(df_no_infra) > 20:
    p = make_panel(df_no_infra)
    m = PanelOLS(p['return_ihs'], sm.add_constant(p[['ai_intensity']]),
                 entity_effects=True, time_effects=True, drop_absorbed=True)
    results_loo['excl_infra'] = m.fit(cov_type='clustered', cluster_entity=True)

df_no_nvda = df_clean[~df_clean['ticker'].isin(['NVDA', 'SMCI'])].copy()
if len(df_no_nvda) > 20:
    p = make_panel(df_no_nvda)
    m = PanelOLS(p['return_ihs'], sm.add_constant(p[['ai_intensity']]),
                 entity_effects=True, time_effects=True, drop_absorbed=True)
    results_loo['excl_NVDA_SMCI'] = m.fit(cov_type='clustered', cluster_entity=True)

# Tabla de robustez
rows = []
for name, res in results_loo.items():
    coef = res.params.get('ai_intensity', None)
    pval = res.pvalues.get('ai_intensity', None)
    ci = res.conf_int().loc['ai_intensity'] if 'ai_intensity' in res.conf_int().index else [None, None]
    rows.append({
        'Especificacion': name,
        'N': int(res.nobs),
        'beta_ai_intensity': round(coef, 5) if coef is not None else None,
        'p_value': round(pval, 4) if pval is not None else None,
        'CI_low': round(ci.iloc[0], 5) if ci is not None and ci.iloc[0] is not None else None,
        'CI_high': round(ci.iloc[1], 5) if ci is not None and ci.iloc[1] is not None else None,
    })

loo_df = pd.DataFrame(rows)
print('=== Analisis de robustez Leave-One-Out ===')
print(loo_df.to_string(index=False))

## 8. Bootstrap: intervalos de confianza (1000 iteraciones)

In [ ]:
# Bootstrap en el coeficiente de ai_intensity (Modelo 1)
# Resampleo a nivel de empresa (cluster bootstrap) para preservar estructura temporal

N_BOOT = 1000
boot_coefs = []

tickers_unique = df_clean['ticker'].unique()

for i in range(N_BOOT):
    # Resamplear empresas con reemplazo
    boot_tickers = np.random.choice(tickers_unique, size=len(tickers_unique), replace=True)
    boot_df = pd.concat([df_clean[df_clean['ticker'] == t].copy() for t in boot_tickers], ignore_index=True)
    # Asignar id unico para que no haya ticker duplicados en el indice
    boot_df['entity_id'] = boot_df.groupby('ticker').ngroup().astype(str) + '_' + \
                           boot_df.groupby('ticker').cumcount().astype(str)
    boot_df = boot_df.set_index(['entity_id', 'year'])

    try:
        m = PanelOLS(
            boot_df['return_ihs'],
            sm.add_constant(boot_df[['ai_intensity']]),
            entity_effects=True, time_effects=True, drop_absorbed=True
        )
        r = m.fit(cov_type='unadjusted')
        coef = r.params.get('ai_intensity', np.nan)
        boot_coefs.append(coef)
    except Exception:
        boot_coefs.append(np.nan)

boot_arr = np.array([c for c in boot_coefs if not np.isnan(c)])
print(f'Bootstrap completado: {len(boot_arr)}/{N_BOOT} iteraciones validas')
print(f'  Coeficiente AI intensity (media bootstrap): {boot_arr.mean():.5f}')
print(f'  IC 95% (percentil):   [{np.percentile(boot_arr, 2.5):.5f}, {np.percentile(boot_arr, 97.5):.5f}]')
print(f'  IC 99%:               [{np.percentile(boot_arr, 0.5):.5f}, {np.percentile(boot_arr, 99.5):.5f}]')

In [ ]:
# Guardar distribucion bootstrap para figura
boot_df_out = pd.DataFrame({'bootstrap_coef_ai_intensity': boot_arr})
boot_df_out.to_csv('../data/processed/bootstrap_ai_intensity.csv', index=False)
print('Distribucion bootstrap guardada')

## 9. Heterogeneidad por grupo

Interacciones AI_intensity x group para detectar si el efecto difiere entre plataformas, SaaS, infra, etc.

In [ ]:
# OLS pooled con interacciones grupo x AI intensity (sin EF para interpretabilidad)
df_het = df_clean.copy()
group_dummies = pd.get_dummies(df_het['group'], prefix='grp', drop_first=True)
df_het = pd.concat([df_het, group_dummies], axis=1)

# Interacciones
for col in group_dummies.columns:
    df_het[f'ai_x_{col}'] = df_het['ai_intensity'] * df_het[col]

inter_cols = [c for c in df_het.columns if c.startswith('ai_x_')]
exog_het = sm.add_constant(df_het[['ai_intensity'] + list(group_dummies.columns) + inter_cols])

mod_het = sm.OLS(df_het['return_ihs'], exog_het).fit(cov_type='HC3')
print('=== Heterogeneidad por grupo (OLS pooled + HC3) ===')
summary_het = pd.DataFrame({
    'coef': mod_het.params,
    'se': mod_het.bse,
    't': mod_het.tvalues,
    'p': mod_het.pvalues
}).round(4)
print(summary_het[summary_het.index.str.contains('ai_')].to_string())

## 10. Resumen de resultados y tabla AEA

## 11. Conclusiones preliminares (v3 — datos mejorados)

**Mejoras de datos respecto a v2:**
- EDGAR CIKs corregidos: SAP, TEAM, PAYC, BRZE, AI (C3.ai), ANET, AVGO (7 empresas recuperadas)
- Soporte IFRS para filers en EUR (SAP)
- `tech_debt_proxy`: 46 empresas (v3) vs 38 (v2) vs 6 (v1)
- Modelo completo: N=180 obs, 32 tickers (v3) vs 151/27 (v2)

---

### H1 — Bifurcacion del mercado (AI intensity → retornos)
- **Modelo 1 (N=437):** β_ai = +0.024, p=0.069* — marginalmente significativo al 10%
- **Interpretacion economica:** +10 puntos en AI intensity ≈ +2.4pp de retorno adicional
- **Quintiles:** Q5 promedia ~36.5% vs Q1 ~21.7% (+14.8pp diferencia bruta)
- **Estado:** H1 soportada debilmente. Con mayor variacion de ai_intensity el efecto puede ser mas claro.

### H2 — Deuda tecnica como moderador
- **Modelo 2 FE (N=180):** β_tech_debt — ver output celda 9
- **Hausman (N=180):** ver celda 13 — si EA preferidos, usar Between-OLS para H2
- **Interpretacion:** la deuda tecnica es mas una caracteristica estructural (nivel entre empresas) que algo que varia ano a ano. El FE within-estimator puede absorber este efecto.
- **Estado:** H2 requiere Between-OLS o RE para capturar variacion cross-sectional.

### H3 — Efecto exogeno ChatGPT (DiD)
- **DiD panel FE (N=437):** ver celda 18-19
- **Parallel Trends (pre-2022):** coef_interaccion = -0.56, p = 0.955 → **SOPORTADO**
  - El supuesto de tendencias paralelas se cumple fuertemente (p>>0.05)
  - Las empresas tratadas y control tenian trayectorias de retorno estadisticamente identicas antes del evento ChatGPT
- **Estado:** DiD identificado causalmente (tendencias paralelas confirmadas)

### Tests de Robustez
- **Bootstrap IC 95%:** ver data/processed/bootstrap_ai_intensity.csv
- **Leave-one-out:** resultados en celda 26 — verificar si NVDA/infra dominan
- **Hausman:** ver celda 13 para veredicto FE vs EA
- **Winsorization:** p95 = 71.0% (excluye SOUN >300%, PRGS write-down 2021)

### Limitaciones (actualizadas)
- `ai_intensity` construida con proxy (producto_binario + rd_norm), no ingresos directos de AI
- `tech_debt_proxy` (COGS/Revenue) captura estructura de costos, no deuda tecnica directa
- BBAI sin datos EDGAR (empresa muy pequena, no XBRL estandarizado)
- SO survey: datos de adopcion AI son estimaciones de reportes publicos, no microdatos
- SAP reporta en EUR (IFRS); los ratios son correctos pero los valores absolutos son comparados con empresas USD

In [ ]:
# Guardar tabla de resultados
results_summary = []
for name, res in models.items():
    for var in res.params.index:
        results_summary.append({
            'model': name,
            'variable': var,
            'coef': round(res.params[var], 6),
            'se': round(res.std_errors[var], 6),
            'pval': round(res.pvalues[var], 6),
            'ci_low': round(res.conf_int().loc[var].iloc[0], 6),
            'ci_high': round(res.conf_int().loc[var].iloc[1], 6),
            'nobs': int(res.nobs),
            'r2_within': round(res.rsquared, 6)
        })

results_df = pd.DataFrame(results_summary)
results_df.to_csv('../data/processed/regression_results.csv', index=False)
print('Resultados guardados en data/processed/regression_results.csv')

# Guardar bootstrap summary
bootstrap_summary = {
    'variable': 'ai_intensity',
    'n_iter': len(boot_arr),
    'mean': round(float(boot_arr.mean()), 6),
    'median': round(float(np.median(boot_arr)), 6),
    'ci_95_low': round(float(np.percentile(boot_arr, 2.5)), 6),
    'ci_95_high': round(float(np.percentile(boot_arr, 97.5)), 6),
    'ci_99_low': round(float(np.percentile(boot_arr, 0.5)), 6),
    'ci_99_high': round(float(np.percentile(boot_arr, 99.5)), 6),
}
pd.DataFrame([bootstrap_summary]).to_csv('../data/processed/bootstrap_summary.csv', index=False)
print('Bootstrap summary guardado en data/processed/bootstrap_summary.csv')

## 11. Conclusiones preliminares

Celda para notas de interpretacion — completar tras revisar resultados numericos.

### H1 (Bifurcacion): 
- Coeficiente `ai_intensity`: [completar]
- Significancia estadistica: [completar]
- Interpretacion economica: un incremento de 10 puntos en AI intensity se asocia a un cambio de X% en el retorno anual

### H2 (Deuda tecnica):
- Coeficiente `tech_debt_proxy`: [completar]
- Signo esperado: negativo

### H3 (Efecto ChatGPT):
- Coeficiente DiD `did_chatgpt`: [completar]
- Test de parallel trends: [completar]

### Robustez:
- Bootstrap IC 95%: [completar]
- Leave-one-out: [completar]
- Hausman test: [completar]

### Limitaciones identificadas:
- N efectivo reducido (~49 obs con ai_intensity completa) — ver Seccion 3 de data_dictionary.md
- AI intensity construida con proxy (no ingresos directos de AI), sesgo de medicion posible
- XBRL coverage parcial: ~50% de empresas con margenes de EDGAR